# ML-KEM 768 — Hardware-Level Benchmark (AXI Timer)

Đo **cycle-accurate** execution time bằng AXI Timer IP nhúng trong FPGA.

| Metric | Software-level | Hardware-level (notebook này) |
|---|---|---|
| Timer | `time.perf_counter()` | AXI Timer (TCR0) |
| Đơn vị | ms (float) | **clock cycles** (integer) |
| Overhead | AXI-Lite + OS jitter | ~0 (timer chạy song song với IP) |
| Precision | ~1μs | **1 clock cycle = 10.3ns** |

## 1. Imports & Configuration

In [ ]:
import numpy as np
import time
import os

from pynq import Overlay, allocate

# Clock frequency from .hwh (pl_clk0)
CLOCK_FREQ_HZ = 96_968_727  # ~96.97 MHz

# Kyber-768 sizes
PK_SIZE   = 1184
SK_SIZE   = 2400
CT_SIZE   = 1088
SS_SIZE   = 32
SEED_SIZE = 32

# AXI-Lite control
REG_CTRL = 0x00
AP_START = 0x01
AP_DONE  = 0x02

# Register offsets
KEYGEN_REG_SEED_D = 0x10
KEYGEN_REG_SEED_Z = 0x1C
KEYGEN_REG_PK_OUT = 0x28
KEYGEN_REG_SK_OUT = 0x34

ENCAPS_REG_PK_IN  = 0x10
ENCAPS_REG_RAND_M = 0x1C
ENCAPS_REG_CT_OUT = 0x28
ENCAPS_REG_SS_OUT = 0x34

DECAPS_REG_SK_IN  = 0x10
DECAPS_REG_CT_IN  = 0x1C
DECAPS_REG_SS_OUT = 0x28

# Bitstream directory
script_dir = os.path.dirname(os.path.abspath("__file__"))
BIT_DIR = os.path.join(script_dir, "..", "bitstream")

print(f"Clock: {CLOCK_FREQ_HZ/1e6:.2f} MHz")
print(f"1 cycle = {1e9/CLOCK_FREQ_HZ:.2f} ns")
print(f"Bitstream dir: {os.path.abspath(BIT_DIR)}")

## 2. Helper Functions

In [ ]:
def _write_pointer(ip, offset_lo, addr):
    """Write a 64-bit physical address into two 32-bit AXI-Lite registers."""
    ip.write(offset_lo, addr & 0xFFFFFFFF)
    ip.write(offset_lo + 4, (addr >> 32) & 0xFFFFFFFF)


def _bytes_to_u64_array(data: bytes) -> np.ndarray:
    assert len(data) % 8 == 0
    return np.frombuffer(data, dtype=np.uint64).copy()


def timer_start(timer):
    """Reset and start AXI Timer 0 (count-up from 0)."""
    timer.write(0x04, 0x00000000)  # TLR0 = 0
    timer.write(0x00, 0x00000020)  # LOAD=1 -> load TLR0 into counter
    timer.write(0x00, 0x00000080)  # LOAD=0, ENT=1 -> start counting up


def timer_stop(timer):
    """Stop AXI Timer 0 and return (cycles, time_ms)."""
    timer.write(0x00, 0x00000000)  # ENT=0 -> halt counter
    cycles = timer.read(0x08)      # TCR0
    hw_time_ms = (cycles / CLOCK_FREQ_HZ) * 1000
    return cycles, hw_time_ms


print("Helpers defined: _write_pointer, _bytes_to_u64_array, timer_start, timer_stop")

## 3. KeyGen Benchmark (AXI Timer)

In [ ]:
print("=" * 60)
print("  KeyGen — Hardware Benchmark (AXI Timer)")
print("=" * 60)

keygen_bit = os.path.join(BIT_DIR, "Keygen", "Hardware_level", "ML_KEM_Kyber_Keygen_1.bit")
ol = Overlay(keygen_bit)
ip = ol.ml_kem_keygen_0
timer = ol.axi_timer_0

seed_d_buf = allocate(shape=(4,), dtype=np.uint64)
seed_z_buf = allocate(shape=(4,), dtype=np.uint64)
pk_buf = allocate(shape=(PK_SIZE,), dtype=np.uint8)
sk_buf = allocate(shape=(SK_SIZE,), dtype=np.uint8)

N_RUNS = 10
keygen_results = []

for run in range(N_RUNS):
    seed_d_buf[:] = _bytes_to_u64_array(os.urandom(32))
    seed_z_buf[:] = _bytes_to_u64_array(os.urandom(32))
    seed_d_buf.flush()
    seed_z_buf.flush()
    _write_pointer(ip, KEYGEN_REG_SEED_D, seed_d_buf.physical_address)
    _write_pointer(ip, KEYGEN_REG_SEED_Z, seed_z_buf.physical_address)
    _write_pointer(ip, KEYGEN_REG_PK_OUT, pk_buf.physical_address)
    _write_pointer(ip, KEYGEN_REG_SK_OUT, sk_buf.physical_address)

    timer_start(timer)
    ip.write(REG_CTRL, AP_START)
    while (ip.read(REG_CTRL) & AP_DONE) == 0: pass
    cycles, hw_ms = timer_stop(timer)
    keygen_results.append((cycles, hw_ms))

    pk_buf.invalidate()
    sk_buf.invalidate()
    print(f"  Run {run+1:2d}/{N_RUNS}: {cycles:>10,} cycles = {hw_ms:.4f} ms")

# Save pk, sk for later use
pk_bytes = bytes(pk_buf)
sk_bytes = bytes(sk_buf)

seed_d_buf.freebuffer()
seed_z_buf.freebuffer()
pk_buf.freebuffer()
sk_buf.freebuffer()
ol.free()

kg_cycles = [r[0] for r in keygen_results]
kg_times  = [r[1] for r in keygen_results]
print(f"\n  Avg: {np.mean(kg_cycles):,.0f} cycles = {np.mean(kg_times):.4f} ms")
print(f"  Min: {np.min(kg_cycles):,} cycles = {np.min(kg_times):.4f} ms")

## 4. Encaps Benchmark (AXI Timer)

In [ ]:
print("=" * 60)
print("  Encaps — Hardware Benchmark (AXI Timer)")
print("=" * 60)

encaps_bit = os.path.join(BIT_DIR, "Encaps", "Hardware_level", "ML_Kyber_Encaps_1.bit")
ol = Overlay(encaps_bit)
ip = ol.ml_kem_encaps_0
timer = ol.axi_timer_0

pk_buf   = allocate(shape=(PK_SIZE,), dtype=np.uint8)
rand_buf = allocate(shape=(SEED_SIZE,), dtype=np.uint8)
ct_buf   = allocate(shape=(CT_SIZE,), dtype=np.uint8)
ss_buf   = allocate(shape=(SS_SIZE,), dtype=np.uint8)

encaps_results = []

for run in range(N_RUNS):
    pk_buf[:] = np.frombuffer(pk_bytes, dtype=np.uint8)
    rand_buf[:] = np.frombuffer(os.urandom(SEED_SIZE), dtype=np.uint8)
    pk_buf.flush()
    rand_buf.flush()
    _write_pointer(ip, ENCAPS_REG_PK_IN,  pk_buf.physical_address)
    _write_pointer(ip, ENCAPS_REG_RAND_M, rand_buf.physical_address)
    _write_pointer(ip, ENCAPS_REG_CT_OUT, ct_buf.physical_address)
    _write_pointer(ip, ENCAPS_REG_SS_OUT, ss_buf.physical_address)

    timer_start(timer)
    ip.write(REG_CTRL, AP_START)
    while (ip.read(REG_CTRL) & AP_DONE) == 0: pass
    cycles, hw_ms = timer_stop(timer)
    encaps_results.append((cycles, hw_ms))

    ct_buf.invalidate()
    ss_buf.invalidate()
    print(f"  Run {run+1:2d}/{N_RUNS}: {cycles:>10,} cycles = {hw_ms:.4f} ms")

ct_bytes = bytes(ct_buf)
ss_enc_bytes = bytes(ss_buf)

pk_buf.freebuffer()
rand_buf.freebuffer()
ct_buf.freebuffer()
ss_buf.freebuffer()
ol.free()

en_cycles = [r[0] for r in encaps_results]
en_times  = [r[1] for r in encaps_results]
print(f"\n  Avg: {np.mean(en_cycles):,.0f} cycles = {np.mean(en_times):.4f} ms")
print(f"  Min: {np.min(en_cycles):,} cycles = {np.min(en_times):.4f} ms")

## 5. Decaps Benchmark (AXI Timer)

In [ ]:
print("=" * 60)
print("  Decaps — Hardware Benchmark (AXI Timer)")
print("=" * 60)

# ĐÚNG (bản mới, có AXI Timer):
decaps_bit = os.path.join(BIT_DIR, "Decaps", "Hardware_level", "ML_KEM_Decaps_wrapper_1.bit")
ol = Overlay(decaps_bit)
ip = ol.ml_kem_decaps_0
timer = ol.axi_timer_0

sk_buf = allocate(shape=(SK_SIZE,), dtype=np.uint8)
ct_buf = allocate(shape=(CT_SIZE,), dtype=np.uint8)
ss_buf = allocate(shape=(SS_SIZE,), dtype=np.uint8)

decaps_results = []

for run in range(N_RUNS):
    sk_buf[:] = np.frombuffer(sk_bytes, dtype=np.uint8)
    ct_buf[:] = np.frombuffer(ct_bytes, dtype=np.uint8)
    sk_buf.flush()
    ct_buf.flush()
    _write_pointer(ip, DECAPS_REG_SK_IN,  sk_buf.physical_address)
    _write_pointer(ip, DECAPS_REG_CT_IN,  ct_buf.physical_address)
    _write_pointer(ip, DECAPS_REG_SS_OUT, ss_buf.physical_address)

    timer_start(timer)
    ip.write(REG_CTRL, AP_START)
    while (ip.read(REG_CTRL) & AP_DONE) == 0: pass
    cycles, hw_ms = timer_stop(timer)
    decaps_results.append((cycles, hw_ms))

    ss_buf.invalidate()
    print(f"  Run {run+1:2d}/{N_RUNS}: {cycles:>10,} cycles = {hw_ms:.4f} ms")

ss_dec_bytes = bytes(ss_buf)

sk_buf.freebuffer()
ct_buf.freebuffer()
ss_buf.freebuffer()
ol.free()

dc_cycles = [r[0] for r in decaps_results]
dc_times  = [r[1] for r in decaps_results]
print(f"\n  Avg: {np.mean(dc_cycles):,.0f} cycles = {np.mean(dc_times):.4f} ms")
print(f"  Min: {np.min(dc_cycles):,} cycles = {np.min(dc_times):.4f} ms")

## 6. Summary & Verification

In [ ]:
print("=" * 68)
print("  SUMMARY — Hardware-Level IP Execution Time (AXI Timer)")
print("=" * 68)
print(f"  {'Kernel':<10} {'Avg Cycles':>12} {'Avg (ms)':>10} {'Min Cycles':>12} {'Min (ms)':>10}")
print(f"  {'-'*10} {'-'*12} {'-'*10} {'-'*12} {'-'*10}")

for name, cyc_list, ms_list in [
    ("KeyGen", kg_cycles, kg_times),
    ("Encaps", en_cycles, en_times),
    ("Decaps", dc_cycles, dc_times),
]:
    print(f"  {name:<10} {np.mean(cyc_list):>12,.0f} {np.mean(ms_list):>10.4f} "
          f"{np.min(cyc_list):>12,} {np.min(ms_list):>10.4f}")

print(f"  {'-'*10} {'-'*12} {'-'*10} {'-'*12} {'-'*10}")
total_cyc = np.mean(kg_cycles) + np.mean(en_cycles) + np.mean(dc_cycles)
total_ms  = np.mean(kg_times)  + np.mean(en_times)  + np.mean(dc_times)
print(f"  {'TOTAL':<10} {total_cyc:>12,.0f} {total_ms:>10.4f}")

print(f"\n  Clock = {CLOCK_FREQ_HZ/1e6:.2f} MHz | N_RUNS = {N_RUNS}")
print(f"  Method: AXI Timer TCR0 [count-up, ENT gate]")

# Verify shared secrets
match = ss_enc_bytes == ss_dec_bytes
print(f"\n  Shared secrets match: {'✅ YES' if match else '❌ NO'}")
if not match:
    print(f"    ss_enc: {ss_enc_bytes.hex()}")
    print(f"    ss_dec: {ss_dec_bytes.hex()}")